In [0]:
# Create dataset 
data = [ (1, 101, '2024-12-15 09:30:00'), (2, 102, '2024-12-15 11:45:00'), (3, 103, '2024-12-15 12:10:00'), (4, 104, '2024-12-15 13:15:00'), (5, 105, '2024-12-15 14:20:00'), (6, 106, '2024-12-15 15:30:00'), (7, 107, '2024-12-15 16:40:00'), (8, 108, '2024-12-16 09:50:00'), (9, 109, '2024-12-16 10:30:00'), (10, 110, '2024-12-16 12:05:00'), (11, 111, '2024-12-16 13:50:00'), (12, 112, '2024-12-16 14:15:00'), (13, 113, '2024-12-16 15:30:00'), (14, 114, '2024-12-17 09:45:00'), (15, 115, '2024-12-17 11:20:00'), (16, 116, '2024-12-17 12:25:00'), (17, 117, '2024-12-17 13:30:00'), (18, 118, '2024-12-17 14:55:00'), (19, 119, '2024-12-17 15:10:00'), (20, 120, '2024-12-18 10:40:00') ]

columns = ["order_id", "product_id", "timestamp"]

df = spark.createDataFrame(data, columns)
df.show()

In [0]:
df.dtypes

In [0]:
from pyspark.sql.functions import *
df = df.withColumn('timestamp',col('timestamp').cast('timestamp'))

In [0]:
df = df.withColumn('day_of_week',date_format(col('timestamp'),"EEEE"))\
       .withColumn('time_of_day',when(hour('timestamp')<12,lit("Morning"))\
                                 .when((hour('timestamp')>12) & (hour('timestamp')<15),lit("Early Afternoon"))\
                                 .otherwise('Late Afternoon')    )
df.show()

In [0]:
df_sum=df.groupBy('day_of_week',"time_of_day").agg(count('order_id').alias('order_count'))
df_sum.show()

In [0]:
from pyspark.sql.window import Window
windows_spec = Window.orderBy(desc(col('order_count')))
df_rank = df_sum.withColumn('rank',rank().over(windows_spec))
df_filter = df_rank.where(col('rank')<=2)
df_filter.show()